# 🍽️ W1: F&B 업종 AI 전환(AX) 자가진단
**hexa-2: 외식/F&B AX 마스터클래스**

---
**학습 목표**
1. 우리 가게의 AI 전환 준비도를 진단한다
2. 현재 디지털화 수준을 객관적으로 파악한다
3. AI 도입을 위한 맞춤형 개선 방안을 제시받는다

> ⏱️ 예상 소요시간: 30분 | 🎯 결과물: AX 자가진단 리포트 MD 파일

## Step 0: 환경 설정

In [ ]:
!pip install -q google-generativeai
import google.generativeai as genai
print('✅ google-generativeai 설치 완료')

In [ ]:
from google.colab import userdata
import os

try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    print('✅ Gemini API 키 로드 성공')
except:
    API_KEY = input('🔑 Gemini API 키를 입력하세요: ')
    genai.configure(api_key=API_KEY)
    print('✅ Gemini API 설정 완료')

## Step 1: 가게 정보 입력 (✏️ 여기만 수정하세요)

In [ ]:
# ✏️ 우리 가게 정보로 수정해주세요
STORE_INFO = {
    'name': '맛있는 분식집',           # 가게명
    'type': '분식/떡볶이',              # 업종 (카페/치킨/한식/분식/일식/중식/양식 등)
    'location': '서울 강남구 역삼동',   # 위치
    'seats': 30,                       # 좌석 수
    'years_open': 5,                   # 운영 연수
    'delivery_apps': ['배달의민족', '요기요']  # 사용 중인 배달앱
}

print('📋 가게 정보 확인')
for key, value in STORE_INFO.items():
    print(f'  • {key}: {value}')

## Step 2: AX 진단 문항

각 문항에 대해 현재 상태를 1~5점으로 평가해주세요.
- **1점**: 전혀 안 함 (수동/종이)
- **2점**: 가끔 함 (불규칙)
- **3점**: 보통 (주기적으로 수행)
- **4점**: 자주 함 (시스템 활용)
- **5점**: 완전 자동화 (AI/시스템 자동)

In [ ]:
# 진단 문항 정의
DIAGNOSTIC_QUESTIONS = [
    {
        'id': 'Q1',
        'category': '메뉴관리',
        'question': '메뉴 업데이트(가격/신메뉴)를 자동화하고 있나요?',
        'description': '메뉴판, 배달앱, SNS 등에 메뉴 변경사항을 자동 반영'
    },
    {
        'id': 'Q2',
        'category': '리뷰관리',
        'question': '배달앱/방문 리뷰를 자동으로 분석하고 답글을 작성하나요?',
        'description': '리뷰 감성 분석, 자동 답글 생성 시스템'
    },
    {
        'id': 'Q3',
        'category': 'SNS운영',
        'question': '인스타그램/블로그 등 SNS 콘텐츠 발행을 자동화하나요?',
        'description': '포스팅 스케줄링, 이미지/글 자동 생성'
    },
    {
        'id': 'Q4',
        'category': '재고관리',
        'question': '식자재 재고를 자동으로 파악하고 발주를 예측하나요?',
        'description': '재고 실시간 추적, 발주량 자동 계산'
    },
    {
        'id': 'Q5',
        'category': '예약관리',
        'question': '예약 시스템을 운영하며 고객에게 자동 알림을 보내나요?',
        'description': '예약 확인, 리마인더 문자 자동 발송'
    },
    {
        'id': 'Q6',
        'category': '매출분석',
        'question': '일일/주간/월간 매출 리포트를 자동 생성하나요?',
        'description': '매출 대시보드, 트렌드 분석 자동화'
    },
    {
        'id': 'Q7',
        'category': '고객관리',
        'question': '단골 고객을 자동으로 식별하고 맞춤 혜택을 제공하나요?',
        'description': 'CRM 시스템, 개인화 마케팅'
    },
    {
        'id': 'Q8',
        'category': '직원관리',
        'question': '직원 근무 스케줄과 급여 계산을 자동화하나요?',
        'description': '스케줄 관리, 급여 자동 계산'
    },
    {
        'id': 'Q9',
        'category': '주문관리',
        'question': '배달/포장/매장 주문을 통합 관리 시스템에서 처리하나요?',
        'description': 'POS 연동, 주문 자동 수집 및 처리'
    },
    {
        'id': 'Q10',
        'category': '데이터활용',
        'question': '영업 데이터를 분석해서 경영 의사결정에 활용하나요?',
        'description': '데이터 기반 메뉴 개발, 가격 정책 수립'
    }
]

print('✅ 진단 문항 로드 완료 (총 10개)')

In [ ]:
# 진단 점수 입력 (테스트용 기본값 - 실제 사용 시 수정)
# ✏️ 각 문항의 점수를 1~5 사이로 수정하세요
scores = {
    'Q1': 2,   # 메뉴 업데이트 자동화
    'Q2': 3,   # 리뷰 분석 & 답글
    'Q3': 2,   # SNS 콘텐츠 자동화
    'Q4': 2,   # 재고 관리 자동화
    'Q5': 3,   # 예약 시스템
    'Q6': 3,   # 매출 리포트 자동화
    'Q7': 2,   # 고객 관리 CRM
    'Q8': 2,   # 직원 관리 자동화
    'Q9': 4,   # 주문 통합 관리
    'Q10': 2   # 데이터 기반 의사결정
}

# 점수 검증
for q_id, score in scores.items():
    if not 1 <= score <= 5:
        raise ValueError(f'{q_id} 점수는 1~5 사이여야 합니다. 현재: {score}')

print('📊 진단 점수 입력 완료')
print('-' * 50)
for q in DIAGNOSTIC_QUESTIONS:
    score = scores[q['id']]
    bar = '■' * score + '□' * (5 - score)
    print(f"{q['id']} [{bar}] {score}점 | {q['category']}")
    print(f"    └ {q['question']}")

## Step 3: AX 레벨 분류

In [ ]:
# 점수 합계 계산
total_score = sum(scores.values())
max_score = len(scores) * 5
percentage = (total_score / max_score) * 100

# 레벨 분류
def get_ax_level(score):
    if score <= 25:
        return {
            'level': '🌱 Starter',
            'range': '10~25점',
            'description': 'AI 전환 초기 단계입니다. 디지털화 기초부터 시작해야 합니다.',
            'color': '🔴'
        }
    elif score <= 40:
        return {
            'level': '🌿 Growing',
            'range': '26~40점',
            'description': '디지털화가 진행 중입니다. 점진적으로 AI를 도입할 준비가 되었습니다.',
            'color': '🟡'
        }
    else:
        return {
            'level': '🌳 AI-Ready',
            'range': '41~50점',
            'description': 'AI 도입 준비가 완료되었습니다. 고급 AI 기능을 바로 활용할 수 있습니다.',
            'color': '🟢'
        }

ax_level = get_ax_level(total_score)

print('═' * 50)
print('📊 AX 자가진단 결과')
print('═' * 50)
print(f"가게명: {STORE_INFO['name']}")
print(f"업종: {STORE_INFO['type']}")
print(f"위치: {STORE_INFO['location']}")
print('-' * 50)
print(f"총점: {total_score}점 / {max_score}점 ({percentage:.1f}%)")
print(f"레벨: {ax_level['color']} {ax_level['level']}")
print(f"범위: {ax_level['range']}")
print('-' * 50)
print(f"진단: {ax_level['description']}")
print('═' * 50)

## Step 4: AI 개선 추천 리포트 생성

In [ ]:
# 점수별 강점/약점 분석
sorted_scores = sorted(scores.items(), key=lambda x: x[1])
weaknesses = [(q_id, score) for q_id, score in sorted_scores if score <= 2]
strengths = [(q_id, score) for q_id, score in sorted_scores if score >= 4]

# 질문 내용 매핑
question_map = {q['id']: q for q in DIAGNOSTIC_QUESTIONS}

print('📉 개선 필요 영역 (2점 이하)')
if weaknesses:
    for q_id, score in weaknesses:
        q = question_map[q_id]
        print(f"  • [{q['category']}] {q['question']}")
        print(f"    현재 점수: {score}점")
else:
    print('  없음 (모든 항목 3점 이상)')

print('\n📈 강점 영역 (4점 이상)')
if strengths:
    for q_id, score in strengths:
        q = question_map[q_id]
        print(f"  • [{q['category']}] {q['question']}")
        print(f"    현재 점수: {score}점")
else:
    print('  없음 (개선 여지가 많습니다)')

In [ ]:
# Gemini API로 맞춤형 개선 리포트 생성
def generate_ai_report():
    model = genai.GenerativeModel('gemini-2.0-flash')
    
    # 약점 영역 목록
    weakness_list = []
    for q_id, score in weaknesses:
        q = question_map[q_id]
        weakness_list.append(f"- [{q['category']}] {q['question']} (현재 {score}점)")
    
    prompt = f"""당신은 F&B 업종 AI 전환 컨설턴트입니다. 다음 가게에 대한 맞춤형 AI 전환 개선 리포트를 작성해주세요.

## 가게 정보
- 가게명: {STORE_INFO['name']}
- 업종: {STORE_INFO['type']}
- 위치: {STORE_INFO['location']}
- 좌석 수: {STORE_INFO['seats']}석
- 운영 연수: {STORE_INFO['years_open']}년
- 배달앱: {', '.join(STORE_INFO['delivery_apps'])}

## 진단 결과
- 총점: {total_score}점 / 50점 ({percentage:.1f}%)
- AX 레벨: {ax_level['level']}

## 개선 필요 영역 (우선순위)
{chr(10).join(weakness_list) if weakness_list else '없음'}

## 요청사항
다음 형식으로 한국어로 작성해주세요:

### 1. 현황 분석
(현재 디지털화 수준과 AI 준비도 분석)

### 2. 우선 개선 추천 (TOP 3)
(가장 효과가 클 개선 항목 3가지와 구체적인 방법)

### 3. 단계별 로드맵
- 1개월: (즉시 시작 가능한 항목)
- 3개월: (중기 목표)
- 6개월: (장기 목표)

### 4. 예상 효과
(시간 절약, 비용 절감, 매출 증대 등)

### 5. 시작하기 팁
(오늘 당장 할 수 있는 액션 아이템)
"""
    
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"⚠️ 리포트 생성 중 오류 발생: {str(e)}\n\nAPI 연결을 확인해주세요."

print('🤖 AI 리포트 생성 중... (약 10~20초 소요)')
ai_report = generate_ai_report()
print('\n' + '='*60)
print(ai_report)
print('='*60)

## Step 5: 결과 Markdown 파일 다운로드

In [ ]:
from datetime import datetime
from google.colab import files

# 마크다운 리포트 생성
report_date = datetime.now().strftime('%Y-%m-%d')

markdown_content = f"""# 🍽️ AX 자가진단 리포트

**{STORE_INFO['name']}** ({STORE_INFO['type']})  
진단일: {report_date}

---

## 📊 진단 결과 요약

| 항목 | 내용 |
|------|------|
| 가게명 | {STORE_INFO['name']} |
| 업종 | {STORE_INFO['type']} |
| 위치 | {STORE_INFO['location']} |
| 좌석 수 | {STORE_INFO['seats']}석 |
| 운영 연수 | {STORE_INFO['years_open']}년 |
| **총점** | **{total_score}점 / 50점 ({percentage:.1f}%)** |
| **AX 레벨** | **{ax_level['color']} {ax_level['level']}** |

---

## 📋 항목별 점수

| 번호 | 카테고리 | 질문 | 점수 |
|------|----------|------|------|
{chr(10).join([f"| {q['id']} | {q['category']} | {q['question']} | {scores[q['id']]}점 |" for q in DIAGNOSTIC_QUESTIONS])}

---

## 🤖 AI 추천 리포트

{ai_report}

---

*본 리포트는 hexa-2 F&B AX 마스터클래스 자가진단 도구로 생성되었습니다.*
*Made with ❤️ by Reasonofmoon × Antigravity*
"""

# 파일 저장
filename = f"AX_진단_{STORE_INFO['name']}_{report_date}.md"
with open(filename, 'w', encoding='utf-8') as f:
    f.write(markdown_content)

# 다운로드
files.download(filename)
print(f'✅ 리포트 다운로드 완료: {filename}')

---
## 🔥 확장 과제
1. **실제 점수로 다시 진단하기**: 위 `scores` 딕셔너리의 값을 실제 상황에 맞게 수정하고 재실행
2. **팀원들과 비교하기**: 같은 업종 다른 가게의 점수와 비교해보기
3. **3개월 후 재진단**: 개선 활동 후 점수 변화 추적하기